# Calibrated interventions, heterogeneity, frequency ablation, sequence sanity (Exp 3, 4, 7, 8)
Turns 'we pushed the answer direction' into a counterfactual with a stated target, reports the calibration-free critical projection m*, ties recovery to trajectory category per model, and checks first-token recovery implies a correct full continuation.

In [ ]:
# --- environment (pins matching the pipeline) ---
# transformers==4.46.2  numpy==1.26.4  ; PyTorch nightly cu128 on newer instances.
import os, gc, json, math, pathlib
import numpy as np, torch
from tqdm.auto import tqdm
import rw_core as rc          # tested core (rw_core_smoketest.py: 19/19)
import rw_modelio as mio      # model IO / hooks / generation

ART = pathlib.Path(os.environ.get("RW_ART", "artifacts")); ART.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def save(obj, name):
    p = ART / name
    np.savez_compressed(p, **obj) if name.endswith(".npz") else \
        p.write_text(json.dumps(obj, indent=2, default=float))
    print("saved", p)

def exists(name):  # skip-if-exists guard
    return (ART / name).exists()


In [ ]:
MODELS = [
    "meta-llama/Llama-3.1-8B", "meta-llama/Llama-3.2-3B", "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-3B-Instruct",
    "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B",
    "mistralai/Mistral-7B-v0.1",
]

In [ ]:
# ============================================================================
# DATA ADAPTER  --  wire this to your existing read/write cache.
# Return a list of item dicts (schema in rw_core docstring). The fields used
# downstream are listed per-cell. This is the ONLY place that knows your layout.
# ============================================================================
def load_items(model_name):
    """TODO: load your per-item records for `model_name`.
    Required keys vary by notebook; each cell asserts what it needs."""
    raise NotImplementedError("wire load_items() to your read/write cache")


In [ ]:
# Build per-item intervention contexts. Needs per readable-but-unselected
# FAILED item: h_final (d,), gold id, alt id; plus W_U per model and freq dir r.
# Successful items supply the calibration distribution of answer-direction support.
from transformers import AutoModelForCausalLM
def get_WU_and_r(name, items):
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16,
                                                 device_map=DEVICE).eval()
    W, _, _ = mio.get_unembedding(model)
    del model; gc.collect(); torch.cuda.empty_cache()
    # frequency direction from this model's own unembedding + logf (cache logf in items)
    logf = items[0]["logf"]                      # (V,) cached once per model
    r = rc.frequency_direction(W, logf)
    return W, r, logf

In [ ]:
# Experiment 3: calibrated targets + dose-response + critical projection.
WG = DEVICE
results = {}
for name in MODELS:
    if exists(f"interv_{name.split('/')[-1]}.json"):
        continue
    items = load_items(name)
    W, r, logf = get_WU_and_r(name, items)
    W_gpu = torch.tensor(W, device=WG)
    succ_pi = []   # answer-direction support of successful items (calibration)
    fail_ctx = []  # contexts for readable-but-unselected failures
    for it in items:
        ctx = rc.build_context(it["h_final"], W_gpu, it["gold_first_tok"],
                               b=it["alt_id"], r=r, device=WG)
        if it["is_direct_success"]:
            succ_pi.append(ctx.pi_a)
        elif it.get("readable_unselected", (not it["is_direct_success"])):
            fail_ctx.append((it, ctx))
    tgt = rc.percentile_targets(succ_pi, qs=(25,50,75))
    targets = sorted(set(list(tgt.values()) +
                         list(np.linspace(min(succ_pi), max(succ_pi), 8))))
    ctxs = [c for _, c in fail_ctx]
    dr_up   = rc.dose_response(ctxs, targets, mode="answer_up", W_U=W_gpu, device=WG)
    # alt-down sweeps the alternative target; both fixes alt at 25th succ pct of alt support
    alt_lo  = float(np.percentile([c.pi_b for c in ctxs], 25))
    dr_both = rc.dose_response(ctxs, targets, mode="both", W_U=W_gpu,
                              alt_target=alt_lo, device=WG)
    dr_rand = rc.dose_response(ctxs, targets, mode="random", W_U=W_gpu, device=WG)
    mstar = [rc.critical_projection(c) for c in ctxs]
    results[name] = {
        "targets": targets, "answer_up": dr_up, "both": dr_both, "random": dr_rand,
        "calib_pct": tgt,
        "mstar_finite_frac": float(np.mean(np.isfinite(mstar))),
        "mstar_median": float(np.nanmedian([m for m in mstar if np.isfinite(m)])),
        "succ_pi_median": float(np.median(succ_pi)),
    }
    save(results[name], f"interv_{name.split('/')[-1]}.json")
    del W_gpu; gc.collect(); torch.cuda.empty_cache()
    print(name, "mstar finite frac", results[name]["mstar_finite_frac"])

In [ ]:
# Dose-response curves per model: recovery vs target projection.
import matplotlib.pyplot as plt
files = sorted(ART.glob("interv_*.json"))
fig, axes = plt.subplots(2, 4, figsize=(18,8), sharey=True)
for ax, fp in zip(axes.ravel(), files):
    d = json.loads(fp.read_text()); t = d["targets"]
    ax.plot(t, [d["answer_up"][str(x)] if str(x) in d["answer_up"] else d["answer_up"][x] for x in t], "o-", label="answer-up")
    ax.plot(t, [d["both"][x] for x in t], "s-", label="both")
    ax.plot(t, [d["random"][x] for x in t], "^--", label="random")
    for q,v in d["calib_pct"].items(): ax.axvline(v, ls=":", alpha=.4)
    ax.set_title(fp.stem.replace("interv_","")); ax.set_xlabel("target projection m")
axes[0,0].set_ylabel("gold recovery"); axes[0,0].legend()
plt.tight_layout(); plt.savefig(ART/"dose_response.png", dpi=140); plt.show()

The **critical projection** `mstar_median` is the calibration-free summary: how much answer-direction support a failure would need to be selected. `mstar_finite_frac < 1` means some failures cannot be fixed by answer-up at any dose — those are competitor-limited, not answer-limited.

In [ ]:
# Experiment 4: tie intervention profile to trajectory category per model.
# Needs delta_layers per readable-unselected item.
per_model_items = {}
interv_rates = {}
for name in MODELS:
    items = [it for it in load_items(name)
             if it.get("readable_unselected", not it["is_direct_success"])
             and "delta_layers" in it]
    per_model_items[name] = items
    d = json.loads((ART/f"interv_{name.split('/')[-1]}.json").read_text())
    # report recovery at the 50th-percentile calibrated target
    m50 = d["calib_pct"]["50"]
    interv_rates[name] = {"answer_up": d["answer_up"][m50] if m50 in d["answer_up"] else d["answer_up"][str(m50)],
                          "both": d["both"][m50], "random": d["random"][m50],
                          "alt_down": float("nan")}
rows = rc.heterogeneity_table(per_model_items, interv_rates)
save({"rows": rows}, "exp4_heterogeneity.json")
import pandas as pd; display(pd.DataFrame(rows).round(3))

**The Qwen2.5-3B finding to write up:** high `alt_dominant_rate` with low `answer_up` but positive `both` => `competitor_limited` large. State it as a result: in alternative-dominant models the binding constraint is competitor support, so answer-up alone fails and only the combined intervention recovers. Drop the word 'most' from the causal claim.

In [ ]:
# Experiment 7: cleaner frequency-direction causal ablation.
abl = []
for name in MODELS:
    items = load_items(name); W, r, logf = get_WU_and_r(name, items)
    W_gpu = torch.tensor(W, device=DEVICE)
    fails = [it for it in items if it.get("readable_unselected", not it["is_direct_success"])]
    changed=rec=fs=rc_ch=rc_rec=0.0; fss=[]
    for i,it in enumerate(fails):
        ctx = rc.build_context(it["h_final"], W_gpu, it["gold_first_tok"],
                               b=it["alt_id"], r=r, device=DEVICE)
        a0 = rc.freq_ablation(ctx, logf, target="zero")
        a1 = rc.freq_ablation_random(ctx, W_gpu, logf, seed=i, device=DEVICE)
        changed += a0["changed"]; rec += a0["gold_recovered"]; fss.append(a0["freq_shift"])
        rc_ch += a1["changed"]; rc_rec += a1["gold_recovered"]
    n=len(fails)
    abl.append({"model":name, "freq_changed":changed/n, "gold_rec":rec/n,
                "df_freq":float(np.mean(fss)), "rand_changed":rc_ch/n, "rand_rec":rc_rec/n})
    del W_gpu; gc.collect(); torch.cuda.empty_cache()
save({"rows":abl}, "exp7_freq_ablation.json")
import pandas as pd; display(pd.DataFrame(abl).round(3))

Conclusion to state: frequency-direction ablation changes the selected token and lowers its frequency, but rarely recovers the gold (`gold_rec` small). Frequency shapes *which competitor* wins; it is not a standalone cause of the dissociation.

In [ ]:
# Experiment 8: does first-token recovery imply a correct full continuation?
# Two checks: (a) among directly-correct items, does greedy decode match an alias
# (validates first-token as a proxy); (b) for intervention-recovered items, force
# the gold first token then greedy-continue and check alias match.
from transformers import AutoTokenizer
seq = []
for name in MODELS[:3]:                       # subset is enough for a sanity check
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16,
                                                 device_map=DEVICE).eval()
    tok = AutoTokenizer.from_pretrained(name)
    items = load_items(name)
    ok_proxy = []
    for it in [x for x in items if x["is_direct_success"]][:200]:
        cont = mio.greedy_continuation(model, tok, it["prompt"])
        norm = rc.normalize_answer(cont)
        ok_proxy.append(any(rc.normalize_answer(a) in norm for a in it["gold_aliases"]))
    seq.append({"model": name, "firsttoken_implies_fullanswer": float(np.mean(ok_proxy)),
                "n": len(ok_proxy)})
    del model; gc.collect(); torch.cuda.empty_cache()
save({"rows": seq}, "exp8_sequence_sanity.json")
print(seq)

If `firsttoken_implies_fullanswer` is high, the first-content-token analysis is meaningful at the sequence level, not just token-local. Report it in a footnote to pre-empt the multi-token objection.